# Notebook 02 — Data Cleaning

**Tier:** Pre-analytics  
**Purpose:** Apply all cleaning transforms and produce the canonical cleaned dataset.  
**Input:** `Data/ncr_ride_bookings.csv` (read-only)  
**Output:** `outputs/cleaned_data.parquet`  
**Caveats:**  
- Nulls in Booking Value / Ride Distance / Payment Method for non-Completed rides are **structural** — they are not imputed.  
- Driver/Customer Ratings nulls are not imputed — missing = ride did not produce a rating.  
- Raw CSV hash is verified before and after to guarantee no modification.

In [4]:
import sys
sys.path.insert(0, '..')

import warnings
import pathlib
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pathlib.Path('outputs').mkdir(exist_ok=True)

## 1. Run Cleaning Pipeline

In [5]:
import os
os.chdir(r"D:\projects\Uber_Data_India_Analytics")
from src.data_loader import load_clean

df = load_clean(force_rebuild=True)
print(f'\nCleaned shape: {df.shape}')

[OK] Row count: 150,000
[OK] 'Booking Value' null alignment correct for cancelled/no-driver statuses.
[OK] 'Ride Distance' null alignment correct for cancelled/no-driver statuses.
[OK] 'Payment Method' null alignment correct for cancelled/no-driver statuses.
[OK] 'Avg VTAT' null alignment: only 'No Driver Found' rows.
  Sample duplicates:
     Booking ID            datetime Booking Status
146  CNR2687237 2024-04-15 19:03:28      Completed
248  CNR5380412 2024-02-01 09:13:13      Completed
317  CNR5071968 2024-10-10 03:56:19      Completed
397  CNR2290384 2024-04-28 01:17:51      Completed
493  CNR3466923 2024-02-11 17:05:28      Completed

[NULL SUMMARY]
                                   null_count  null_pct
Incomplete Rides                       141000      94.0
Incomplete Rides Reason                141000      94.0
Cancelled Rides by Customer            139500      93.0
Reason for cancelling by Customer      139500      93.0
Cancelled Rides by Driver              123000      82.0
D

## 2. Dtypes After Cleaning

In [6]:
print(df.dtypes.to_string())

Date                                         object
Time                                         object
Booking ID                                   object
Booking Status                             category
Customer ID                                  object
Vehicle Type                               category
Pickup Location                              object
Drop Location                                object
Avg VTAT                                    float64
Avg CTAT                                    float64
Cancelled Rides by Customer                    Int8
Reason for cancelling by Customer            object
Cancelled Rides by Driver                      Int8
Driver Cancellation Reason                   object
Incomplete Rides                               Int8
Incomplete Rides Reason                      object
Booking Value                               float64
Ride Distance                               float64
Driver Ratings                              float64
Customer Rat

## 3. Null Summary After Cleaning

In [7]:
null_counts = df.isna().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_df = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_df = null_df[null_df['null_count'] > 0].sort_values('null_count', ascending=False)
print('Remaining nulls (all are structurally expected):')
print(null_df.to_string())

Remaining nulls (all are structurally expected):
                                   null_count  null_pct
Incomplete Rides                       141000      94.0
Incomplete Rides Reason                141000      94.0
Cancelled Rides by Customer            139500      93.0
Reason for cancelling by Customer      139500      93.0
Cancelled Rides by Driver              123000      82.0
Driver Cancellation Reason             123000      82.0
Driver Ratings                          57000      38.0
Customer Rating                         57000      38.0
Avg CTAT                                48000      32.0
Ride Distance                           48000      32.0
Booking Value                           48000      32.0
Payment Method                          48000      32.0
Avg VTAT                                10500       7.0


## 4. Verify No "null" Strings Remain

In [8]:
str_cols = df.select_dtypes(include='object').columns
null_str_remaining = {c: (df[c] == 'null').sum() for c in str_cols if (df[c] == 'null').sum() > 0}
if null_str_remaining:
    print(f'⚠ Remaining "null" strings: {null_str_remaining}')
else:
    print('✓ No "null" strings remain in any column.')

✓ No "null" strings remain in any column.


## 5. Verify Booking ID Strip

In [9]:
triple_q = df['Booking ID'].str.startswith('"', na=False).sum()
print(f'Booking IDs still containing quotes: {triple_q}')
print('Sample Booking IDs after cleaning:')
print(df['Booking ID'].head(5).tolist())

Booking IDs still containing quotes: 0
Sample Booking IDs after cleaning:
['CNR5884300', 'CNR1326809', 'CNR8494506', 'CNR8906825', 'CNR1950162']


## 6. Structural Null Alignment Check

In [10]:
print('Booking Value non-null by status (expect: Completed only):')
print(df.dropna(subset=['Booking Value']).groupby('Booking Status', observed=True).size())

print('\nPayment Method non-null by status (expect: Completed only):')
print(df.dropna(subset=['Payment Method']).groupby('Booking Status', observed=True).size())

print('\nAvg VTAT null by status (expect: No Driver Found only):')
print(df[df['Avg VTAT'].isna()].groupby('Booking Status', observed=True).size())

Booking Value non-null by status (expect: Completed only):
Booking Status
Completed     93000
Incomplete     9000
dtype: int64

Payment Method non-null by status (expect: Completed only):
Booking Status
Completed     93000
Incomplete     9000
dtype: int64

Avg VTAT null by status (expect: No Driver Found only):
Booking Status
No Driver Found    10500
dtype: int64


## 7. Observed Numeric Ranges (Post-Clean)

In [11]:
num_cols = ['Avg VTAT', 'Avg CTAT', 'Booking Value', 'Ride Distance', 'Driver Ratings', 'Customer Rating']
print(df[num_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(2).to_string())

        Avg VTAT   Avg CTAT  Booking Value  Ride Distance  Driver Ratings  Customer Rating
count  139500.00  102000.00      102000.00      102000.00        93000.00         93000.00
mean        8.46      29.15         508.30          24.64            4.23             4.40
std         3.77       8.90         395.81          14.00            0.44             0.44
min         2.00      10.00          50.00           1.00            3.00             3.00
25%         5.30      21.60         234.00          12.46            4.10             4.20
50%         8.30      28.80         414.00          23.72            4.30             4.50
75%        11.30      36.80         689.00          36.82            4.60             4.80
95%        14.60      43.40        1224.00          47.35            4.90             5.00
max        20.00      45.00        4277.00          50.00            5.00             5.00


## 8. Completed Rides with Null Booking Value (Integrity Violations)

In [12]:
import pathlib
q_path = pathlib.Path('outputs/completed_nullfare_quarantine.csv')
if q_path.exists():
    quarantined = pd.read_csv(q_path)
    print(f'⚠ Quarantined rows (Completed + null Booking Value): {len(quarantined)}')
    print(quarantined.head())
else:
    print('✓ No data integrity violations found (no quarantine file created).')

✓ No data integrity violations found (no quarantine file created).


## 9. Sample of Cleaned Data

In [13]:
df.head(5)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,datetime
0,2024-03-23,12:29:38,CNR5884300,No Driver Found,CID1982111,eBike,Palam Vihar,Jhilmil,NaN,NaN,...,<NA>,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38
1,2024-11-29,18:01:39,CNR1326809,Incomplete,CID4604802,Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,<NA>,NaN,1,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39
2,2024-08-23,08:56:10,CNR8494506,Completed,CID9202816,Auto,Khandsa,Malviya Nagar,13.4,25.8,...,<NA>,NaN,<NA>,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10
3,2024-10-21,17:17:25,CNR8906825,Completed,CID2610914,Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,<NA>,NaN,<NA>,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25
4,2024-09-16,22:08:00,CNR1950162,Completed,CID9933542,Bike,Ghitorni Village,Khan Market,5.3,19.6,...,<NA>,NaN,<NA>,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00


## Limitations
- Nulls in post-outcome columns (Booking Value, Ride Distance, etc.) for non-Completed rides are intentionally retained as NaN; they are never imputed.
- No rows are dropped from the cleaned dataset (except those in the integrity quarantine).
- No outlier removal is performed at this stage; outlier investigation occurs in Notebook 03.